<a href="https://colab.research.google.com/github/hwasun-zip/SOC-Alert-Triage/blob/main/SOC_Triage_CICIDS2017.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1단계: 환경 준비와 데이터 다운로드

In [1]:
# 패키지 설치: DuckDB는 파일 하나로 도는 분석용 SQL 엔진
!pip -q install duckdb anthropic huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 26.2 MB/s eta 0:00:00


In [2]:
# 설정값: 탐지 규칙의 임계값을 맨 위에 모아둔 이유는 나중에 값을 바꿔가며 "경보량 vs 포착률"을 비교하기 쉽게 하려고
import os, re, glob, json, zipfile, random
import numpy as np, pandas as pd, duckdb
import matplotlib.pyplot as plt

DATA_DIR = "/content/cicids2017"
DB_PATH  = "/content/soc.duckdb"
OUT_DIR  = "/content/soc_out"
os.makedirs(OUT_DIR, exist_ok=True)

BASELINE_KEYWORD = "Monday"            # 정상 프로파일 학습용 파일(정상 트래픽만 존재)

# 탐지 규칙 임계값 (라벨을 보지 않고 공격 행위의 특성으로 정함)
SCAN_MIN_PORTS     = 50                 # R1: 1분 동안 한 출발지→목적지가 두드린 서로 다른 포트 수
BF_PORTS           = (21, 22, 23, 3389) # R2: 인증 서비스 포트(FTP, SSH, Telnet, RDP)
BF_MIN_FLOWS       = 10                 # R2: 1분 동안 같은 인증 포트로 반복 접속한 횟수
FLOOD_MIN_FLOWS    = 300                # R3: 1분 동안 한 목적지 서비스로 들어온 최소 플로우 수
FLOOD_K            = 5                  # R3: 정상일(월요일) 분당 p99 대비 배수
SLOW_MIN_FLOWS     = 20                 # R4: 1분 동안 열린 '길고 거의 빈' 연결 수
SLOW_MIN_DUR_US    = 5_000_000          # R4: 연결 유지 시간 5초 이상
SLOW_MAX_FWD_BYTES = 1_000              # R4: 그 동안 보낸 데이터 1KB 이하

# LLM 트리아지
ANTHROPIC_MODEL   = "claude-sonnet-5-5"
MAX_LLM_INCIDENTS = 40
SEED = 42

In [3]:
# 데이터 다운로드: IP·시간 정보가 있는 GeneratedLabelledFlows 버전을 받음 -> 머신러닝용 CSV에는 이 정보가 없어서 "누가 누구에게 언제"를 묶는 규칙을 만들 수 없음
def find_csvs():
    return sorted(p for p in glob.glob(f"{DATA_DIR}/**/*.csv", recursive=True)
                  if not os.path.basename(p).startswith("._"))

os.makedirs(DATA_DIR, exist_ok=True)
csvs = find_csvs()
if not csvs:
    from huggingface_hub import hf_hub_download
    zp = hf_hub_download(repo_id="bencorn/CICIDS2017", repo_type="dataset",
                         filename="csvs/GeneratedLabelledFlows.zip")
    with zipfile.ZipFile(zp) as z:
        z.extractall(DATA_DIR)
    csvs = find_csvs()

for p in csvs:
    print(f"{os.path.getsize(p)/1e6:7.1f} MB  {os.path.basename(p)}")

csvs/GeneratedLabelledFlows.zip: reconstructing file:   0%|          |  0.00B /  284MB            

csvs/GeneratedLabelledFlows.zip: downloading bytes:           |  0.00B            

   96.1 MB  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  101.9 MB  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
   75.4 MB  Friday-WorkingHours-Morning.pcap_ISCX.csv
  268.6 MB  Monday-WorkingHours.pcap_ISCX.csv
  108.7 MB  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
   92.0 MB  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  174.7 MB  Tuesday-WorkingHours.pcap_ISCX.csv
  285.6 MB  Wednesday-workingHours.pcap_ISCX.csv


In [ ]:
# 2단계: 정제와 적재

In [4]:
# 정제 및 적재: 파일이 커서 50만 행씩 나눠 읽고, 필요한 14개 컬럼만 골라 DuckDB에 쌓음

NEED = {"Source IP":"src_ip", "Source Port":"src_port", "Destination IP":"dst_ip",
        "Destination Port":"dst_port", "Protocol":"proto", "Timestamp":"ts_raw",
        "Flow Duration":"duration_us", "Total Fwd Packets":"fwd_pkts", "Total Backward Packets":"bwd_pkts",
        "Total Length of Fwd Packets":"fwd_bytes", "Total Length of Bwd Packets":"bwd_bytes",
        "SYN Flag Count":"syn", "RST Flag Count":"rst", "Label":"label"}
NUM = ["src_port","dst_port","proto","duration_us","fwd_pkts","bwd_pkts","fwd_bytes","bwd_bytes","syn","rst"]

def clean_label(s):
    s = re.sub(r"[^\x20-\x7E]+", "-", str(s)).strip()   # 깨진 문자 → '-'
    return re.sub(r"\s+", " ", s)

def parse_ts(s):
    s = s.str.strip()
    ts = pd.to_datetime(s, format="%d/%m/%Y %H:%M:%S", errors="coerce")   # 초까지 있는 형식
    miss = ts.isna()
    ts[miss] = pd.to_datetime(s[miss], format="%d/%m/%Y %H:%M", errors="coerce")  # 분까지만 있는 형식
    return ts.where(ts.dt.hour >= 8, ts + pd.Timedelta(hours=12))          # 12시간제 보정

con = duckdb.connect(DB_PATH)
con.execute("DROP TABLE IF EXISTS flows")
fid0, created = 0, False
for path in csvs:
    fname = os.path.basename(path)
    n_file = 0
    for ch in pd.read_csv(path, encoding="latin-1", dtype=str, chunksize=500_000,
                          usecols=lambda c: c.strip() in NEED):
        ch.columns = [NEED[c.strip()] for c in ch.columns]
        ch = ch.dropna(subset=["label", "ts_raw"])                 # 빈 행 제거
        if ch.empty:
            continue
        for c in NUM:
            ch[c] = pd.to_numeric(ch[c], errors="coerce").replace([np.inf, -np.inf], np.nan).astype("float64")
        ch["src_ip"] = ch["src_ip"].str.strip()
        ch["dst_ip"] = ch["dst_ip"].str.strip()
        ch["ts"] = parse_ts(ch["ts_raw"])
        ch["m"] = ch["ts"].dt.floor("min")                         # 1분 단위 윈도우 키
        ch["day"] = fname.split("-")[0].capitalize()
        ch["src_file"] = fname
        ch["is_baseline"] = BASELINE_KEYWORD.lower() in fname.lower()
        ch["label"] = ch["label"].map(clean_label)
        ch["is_attack"] = ch["label"].str.upper() != "BENIGN"
        ch["fid"] = np.arange(fid0, fid0 + len(ch)); fid0 += len(ch)   # 플로우 고유 ID
        ch = ch.drop(columns="ts_raw")
        con.register("ch", ch)
        if not created:
            con.execute("CREATE TABLE flows AS SELECT * FROM ch"); created = True
        else:
            con.execute("INSERT INTO flows BY NAME SELECT * FROM ch")
        con.unregister("ch")
        n_file += len(ch)
    print(f"{fname:<62} {n_file:>10,} rows")

con.execute('''CREATE OR REPLACE TABLE flows AS
               SELECT * REPLACE (CAST(dst_port AS INTEGER) AS dst_port,
                                 CAST(src_port AS INTEGER) AS src_port,
                                 CAST(proto    AS INTEGER) AS proto)
               FROM flows WHERE ts IS NOT NULL''')
total = con.sql("SELECT COUNT(*) FROM flows").fetchone()[0]
print(f"\n적재 완료: {total:,} flows (시간 파싱 실패로 제외: {fid0 - total:,})")

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv                  225,745 rows
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv              286,467 rows
Friday-WorkingHours-Morning.pcap_ISCX.csv                         191,033 rows
Monday-WorkingHours.pcap_ISCX.csv                                 529,918 rows
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv       288,602 rows
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv            170,366 rows
Tuesday-WorkingHours.pcap_ISCX.csv                                445,909 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wednesday-workingHours.pcap_ISCX.csv                              692,703 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


적재 완료: 2,830,743 flows (시간 파싱 실패로 제외: 0)


In [5]:
# 시간 보정 검증 (공격 타임라인): 공격이 공개된 일정(화~금, 오전·오후)에 맞게 놓였는지 확인

con.sql('''
    SELECT day, label, COUNT(*) AS flows,
           MIN(m) AS start_m, MAX(m) AS end_m,
           date_diff('minute', MIN(m), MAX(m)) + 1 AS minutes
    FROM flows
    GROUP BY day, label
    ORDER BY MIN(m)''').df()

,day,label,flows,start_m,end_m,minutes
0,Monday,BENIGN,529918,2017-07-03 08:55:00,2017-07-03 17:01:00,487
1,Tuesday,BENIGN,432074,2017-07-04 08:53:00,2017-07-04 17:00:00,488
2,Tuesday,FTP-Patator,7938,2017-07-04 09:17:00,2017-07-04 10:30:00,74
3,Tuesday,SSH-Patator,5897,2017-07-04 14:09:00,2017-07-04 15:11:00,63
4,Wednesday,BENIGN,440031,2017-07-05 08:42:00,2017-07-05 17:10:00,509
5,Wednesday,DoS slowloris,5796,2017-07-05 09:01:00,2017-07-05 14:25:00,325
6,Wednesday,DoS Slowhttptest,5499,2017-07-05 10:15:00,2017-07-05 10:37:00,23
7,Wednesday,DoS Hulk,231073,2017-07-05 10:43:00,2017-07-05 11:07:00,25
8,Wednesday,DoS GoldenEye,10293,2017-07-05 11:10:00,2017-07-05 11:19:00,10
9,Wednesday,Heartbleed,11,2017-07-05 15:12:00,2017-07-05 15:32:00,21


In [ ]:
# 3단계: 행위 기반 SQL 탐지 규칙

In [6]:
# 정상 프로파일 학습 + 탐지 규칙 실행: 결과는 두 테이블에 저장. alert_groups는 "어느 1분에 어느 대상이 걸렸나", flow_hits는 "그 경보에 해당하는 개별 플로우"

# R3용 정상 프로파일: 월요일 기준, 목적지 서비스별 분당 연결 수의 p99
con.execute('''
CREATE OR REPLACE TABLE flood_baseline AS
WITH per_min AS (
    SELECT dst_ip, dst_port, m, COUNT(*) AS c
    FROM flows WHERE is_baseline
    GROUP BY ALL)
SELECT dst_ip, dst_port, quantile_cont(c, 0.99) AS p99, MAX(c) AS mx, COUNT(*) AS active_minutes
FROM per_min GROUP BY ALL''')

bf_ports = "(" + ",".join(map(str, BF_PORTS)) + ")"
RULES = [
    dict(name="R1_PORT_SCAN", keys=["src_ip", "dst_ip"],
         row_filter="f.fwd_pkts <= 3",
         sql=f'''SELECT src_ip, dst_ip, m, COUNT(*) AS n_flows FROM flows
                 WHERE NOT is_baseline AND fwd_pkts <= 3
                 GROUP BY src_ip, dst_ip, m
                 HAVING COUNT(DISTINCT dst_port) >= {SCAN_MIN_PORTS}'''),
    dict(name="R2_BRUTE_FORCE", keys=["src_ip", "dst_ip", "dst_port"],
         row_filter="TRUE",
         sql=f'''SELECT src_ip, dst_ip, dst_port, m, COUNT(*) AS n_flows FROM flows
                 WHERE NOT is_baseline AND dst_port IN {bf_ports}
                 GROUP BY src_ip, dst_ip, dst_port, m
                 HAVING COUNT(*) >= {BF_MIN_FLOWS}'''),
    dict(name="R3_VOLUME_FLOOD", keys=["dst_ip", "dst_port"],
         row_filter="TRUE",
         sql=f'''WITH c AS (SELECT dst_ip, dst_port, m, COUNT(*) AS n_flows FROM flows
                            WHERE NOT is_baseline GROUP BY dst_ip, dst_port, m)
                 SELECT c.* FROM c LEFT JOIN flood_baseline b USING (dst_ip, dst_port)
                 WHERE c.n_flows >= {FLOOD_MIN_FLOWS}
                   AND c.n_flows > {FLOOD_K} * COALESCE(b.p99, 0)'''),
    dict(name="R4_SLOW_DOS", keys=["src_ip", "dst_ip", "dst_port"],
         row_filter=f"f.duration_us >= {SLOW_MIN_DUR_US} AND f.fwd_bytes <= {SLOW_MAX_FWD_BYTES}",
         sql=f'''SELECT src_ip, dst_ip, dst_port, m, COUNT(*) AS n_flows FROM flows
                 WHERE NOT is_baseline AND duration_us >= {SLOW_MIN_DUR_US} AND fwd_bytes <= {SLOW_MAX_FWD_BYTES}
                 GROUP BY src_ip, dst_ip, dst_port, m
                 HAVING COUNT(*) >= {SLOW_MIN_FLOWS}'''),
]

def run_rules():
    con.execute('''CREATE OR REPLACE TABLE alert_groups (gid BIGINT, rule VARCHAR, src_ip VARCHAR,
                   dst_ip VARCHAR, dst_port INTEGER, m TIMESTAMP, n_flows BIGINT)''')
    con.execute("CREATE OR REPLACE TABLE flow_hits (fid BIGINT, gid BIGINT, rule VARCHAR)")
    for r in RULES:
        # 1) 규칙에 걸린 (대상, 1분) 윈도우 저장
        con.execute(f'''INSERT INTO alert_groups BY NAME
                        SELECT (SELECT COALESCE(MAX(gid), 0) FROM alert_groups) + row_number() OVER () AS gid,
                               '{r["name"]}' AS rule, q.*
                        FROM ({r["sql"]}) q''')
        # 2) 그 윈도우에 속한 개별 플로우에 경보 표시
        on = " AND ".join([f"f.{k} = g.{k}" for k in r["keys"]] + ["f.m = g.m"])
        con.execute(f'''INSERT INTO flow_hits
                        SELECT f.fid, g.gid, g.rule FROM flows f
                        JOIN alert_groups g ON g.rule = '{r["name"]}' AND {on}
                        WHERE NOT f.is_baseline AND ({r["row_filter"]})''')

run_rules()
print("탐지 규칙 실행 완료")

탐지 규칙 실행 완료


In [7]:
# 규칙별 경보량 확인
con.sql('''
    SELECT g.rule,
           COUNT(DISTINCT g.gid) AS alert_windows,   -- 걸린 (대상, 1분) 윈도우 수
           COUNT(DISTINCT h.fid) AS flows_hit        -- 경보가 붙은 플로우 수
    FROM alert_groups g LEFT JOIN flow_hits h USING (gid)
    GROUP BY g.rule ORDER BY g.rule''').df()

,rule,alert_windows,flows_hit
0,R1_PORT_SCAN,182,252903
1,R2_BRUTE_FORCE,159,14648
2,R3_VOLUME_FLOOD,63,379220
3,R4_SLOW_DOS,735,228754


In [ ]:
# 4단계: 채점 (처음으로 라벨 사용)

In [8]:
# 전체 성적표 + 공격 유형별 포착률

# 경보율: 전체 플로우 중 도장이 찍힌 비율 → 보안 담당자가 봐야 할 양
# 포착률: 공격 플로우 중 도장이 찍힌 비율 → 공격을 얼마나 잡았나
# 정밀도: 도장 찍힌 것 중 진짜 공격 비율 → 경보를 얼마나 믿을 수 있나
# Lift: 포착률 ÷ 경보율 → 아무거나 찍었을 때보다 몇 배 효율적인가

con.execute('''
CREATE OR REPLACE TABLE flow_eval AS
SELECT f.fid, f.day, f.label, f.is_attack, f.m, (h.fid IS NOT NULL) AS alerted
FROM flows f LEFT JOIN (SELECT DISTINCT fid FROM flow_hits) h USING (fid)
WHERE NOT f.is_baseline''')

n, attacks, alerts, tp = con.sql('''
    SELECT COUNT(*), SUM(is_attack::INT), SUM(alerted::INT), SUM((alerted AND is_attack)::INT)
    FROM flow_eval''').fetchone()
alert_rate, recall = alerts / n, tp / attacks
precision = tp / alerts if alerts else 0.0
lift = recall / alert_rate
KPI = dict(flows=n, attack_flows=attacks, alert_rate=alert_rate, recall=recall, precision=precision, lift=lift)
print(f"평가 플로우 {n:,}건 (공격 {attacks:,}건)")
print(f"경보율 {alert_rate:.1%} | 포착률 {recall:.1%} | 정밀도 {precision:.1%} | Lift {lift:.1f}배\n")

by_label = con.sql('''
    SELECT label, COUNT(*) AS flows, SUM(alerted::INT) AS alerted,
           ROUND(AVG(alerted::INT), 4) AS alert_ratio
    FROM flow_eval GROUP BY label, is_attack
    ORDER BY is_attack DESC, flows DESC''').df()
print(by_label.to_string())

평가 플로우 2,300,825건 (공격 557,646건)
경보율 29.1% | 포착률 98.7% | 정밀도 82.3% | Lift 3.4배

                         label    flows   alerted  alert_ratio
0                     DoS Hulk   231073  230850.0       0.9990
1                     PortScan   158930  158481.0       0.9972
2                         DDoS   128027  128027.0       1.0000
3                DoS GoldenEye    10293   10293.0       1.0000
4                  FTP-Patator     7938    7936.0       0.9997
5                  SSH-Patator     5897    5897.0       1.0000
6                DoS slowloris     5796    2349.0       0.4053
7             DoS Slowhttptest     5499    4796.0       0.8722
8                          Bot     1966       0.0       0.0000
9     Web Attack - Brute Force     1507    1263.0       0.8381
10            Web Attack - XSS      652     588.0       0.9018
11                Infiltration       36       0.0       0.0000
12  Web Attack - Sql Injection       21       0.0       0.0000
13                  Heartbleed       11

In [9]:
# 규칙별로 "실제로 뭘 잡았나" 뜯어보기: R1과 R4가 공격보다 훨씬 많이 울린 이유를 확인

rule_x_label = con.sql('''
    SELECT h.rule, e.label, COUNT(DISTINCT h.fid) AS flows
    FROM flow_hits h JOIN flow_eval e USING (fid)
    GROUP BY h.rule, e.label
    ORDER BY h.rule, flows DESC''').df()
print(rule_x_label.to_string())

               rule                     label   flows
0      R1_PORT_SCAN                  PortScan  158396
1      R1_PORT_SCAN                    BENIGN   94504
2      R1_PORT_SCAN                      DDoS       3
3    R2_BRUTE_FORCE               FTP-Patator    7936
4    R2_BRUTE_FORCE               SSH-Patator    5897
5    R2_BRUTE_FORCE                  PortScan     643
6    R2_BRUTE_FORCE                    BENIGN     172
7   R3_VOLUME_FLOOD                  DoS Hulk  230726
8   R3_VOLUME_FLOOD                      DDoS  128024
9   R3_VOLUME_FLOOD             DoS GoldenEye   10293
10  R3_VOLUME_FLOOD          DoS Slowhttptest    4756
11  R3_VOLUME_FLOOD                    BENIGN    4015
12  R3_VOLUME_FLOOD             DoS slowloris    1406
13      R4_SLOW_DOS                  DoS Hulk  136824
14      R4_SLOW_DOS                      DDoS   51049
15      R4_SLOW_DOS                    BENIGN   20057
16      R4_SLOW_DOS             DoS GoldenEye    9391
17      R4_SLOW_DOS         

In [ ]:
# 5단계: 오탐 원인 진단

In [10]:
# R1 헛경보 들여다보기: R1에 걸린 플로우를 "누가 누구에게"로 묶어서 정상과 공격을 나란히 비교

r1 = con.sql('''
    SELECT f.label, f.src_ip, f.dst_ip,
           COUNT(*) AS flows,
           COUNT(DISTINCT f.dst_port) AS n_dst_ports,
           MIN(f.dst_port) AS min_dport, MAX(f.dst_port) AS max_dport,
           mode(f.src_port) AS top_src_port,
           ROUND(AVG((f.src_port < 1024)::INT), 3) AS src_wellknown
    FROM flow_hits h JOIN flows f USING (fid)
    WHERE h.rule = 'R1_PORT_SCAN'
    GROUP BY f.label, f.src_ip, f.dst_ip
    ORDER BY flows DESC
    LIMIT 15''').df()
print(r1.to_string())

       label          src_ip         dst_ip   flows  n_dst_ports  min_dport  max_dport  top_src_port  src_wellknown
0   PortScan      172.16.0.1  192.168.10.50  158396          998          1      65389         59328          0.000
1     BENIGN   192.168.10.50     172.16.0.1   20072        17523       5117      65532            80          0.982
2     BENIGN    192.168.10.8   192.168.10.5   12122         1006          1      65389         44548          0.000
3     BENIGN    192.168.10.8  192.168.10.15    8084         1008          1      65389         62905          0.000
4     BENIGN    192.168.10.8  192.168.10.14    8060         1006          1      65389         56148          0.000
5     BENIGN    192.168.10.8  192.168.10.25    6162         1001          1      65389         42484          0.000
6     BENIGN    192.168.10.8   192.168.10.9    5326          999          1      65389         60467          0.000
7     BENIGN    192.168.10.8  192.168.10.50    4279         1006        

In [11]:
# R4와 slowloris 들여다보기: 슬로우 공격이 정상 트래픽과 비교해서 "연결 시간"과 "보낸 데이터 양"이 실제로 어떻게 다른지 보기

prof = con.sql('''
    SELECT label, COUNT(*) AS flows, mode(dst_port) AS top_dport,
           quantile_cont(duration_us / 1e6, [0.1, 0.5, 0.9]) AS duration_sec,
           quantile_cont(fwd_bytes,          [0.1, 0.5, 0.9]) AS fwd_bytes,
           quantile_cont(fwd_pkts,           [0.1, 0.5, 0.9]) AS fwd_pkts
    FROM flows
    WHERE NOT is_baseline
      AND label IN ('BENIGN', 'DoS slowloris', 'DoS Slowhttptest', 'DoS Hulk')
    GROUP BY label''').df()
print(prof.to_string())

r4_fp = con.sql('''
    SELECT f.dst_ip, f.dst_port, COUNT(*) AS flows, COUNT(DISTINCT f.src_ip) AS n_src
    FROM flow_hits h JOIN flows f USING (fid)
    WHERE h.rule = 'R4_SLOW_DOS' AND f.label = 'BENIGN'
    GROUP BY f.dst_ip, f.dst_port
    ORDER BY flows DESC LIMIT 10''').df()
print("\nR4 헛경보가 몰린 목적지 Top 10")
print(r4_fp.to_string())

              label    flows  top_dport                                         duration_sec            fwd_bytes          fwd_pkts
0     DoS slowloris     5796         80                   [0.0004485, 97.082009, 105.653647]  [0.0, 16.0, 2541.0]  [1.0, 3.0, 15.0]
1          DoS Hulk   231073         80                         [3e-06, 84.92919, 99.905788]  [0.0, 334.0, 412.0]   [2.0, 6.0, 9.0]
2            BENIGN  1743179         53                      [4.8e-05, 0.030886, 60.3574352]  [0.0, 66.0, 1048.0]  [1.0, 2.0, 14.0]
3  DoS Slowhttptest     5499         80  [0.1561248000000002, 63.117636, 109.46288320000001]   [0.0, 0.0, 1780.0]   [1.0, 7.0, 7.0]

R4 헛경보가 몰린 목적지 Top 10
           dst_ip  dst_port  flows  n_src
0    192.168.10.1        53  15761      1
1    192.168.10.3        53   1237      5
2  162.208.20.178        80    546      3
3    23.60.139.27        80    257      6
4   23.111.11.111       443    189      1
5    23.15.155.27        80    180      5
6     23.50.75.27      

In [ ]:
# 6단계: 짐작을 증거로 확인

In [12]:
# 192.168.10.8 스캔 시점 vs Infiltration 시점 비교

print("① Infiltration 라벨이 붙은 플로우의 시간대")
print(con.sql('''
    SELECT src_ip, dst_ip, COUNT(*) AS flows, MIN(m) AS first_m, MAX(m) AS last_m
    FROM flows WHERE label = 'Infiltration'
    GROUP BY src_ip, dst_ip ORDER BY first_m''').df().to_string())

print("\n② R1이 잡은 '정상' 라벨 스캔의 시간대 (출발지별)")
print(con.sql('''
    SELECT f.day, f.src_ip,
           COUNT(*) AS flows, COUNT(DISTINCT f.dst_ip) AS n_targets,
           MIN(f.m) AS first_m, MAX(f.m) AS last_m,
           ROUND(AVG((f.src_port < 1024)::INT), 3) AS src_wellknown
    FROM flow_hits h JOIN flows f USING (fid)
    WHERE h.rule = 'R1_PORT_SCAN' AND f.label = 'BENIGN'
    GROUP BY f.day, f.src_ip
    ORDER BY flows DESC LIMIT 10''').df().to_string())

① Infiltration 라벨이 붙은 플로우의 시간대
         src_ip          dst_ip  flows             first_m              last_m
0  192.168.10.8  205.174.165.73     36 2017-07-06 14:19:00 2017-07-06 15:45:00

② R1이 잡은 '정상' 라벨 스캔의 시간대 (출발지별)
         day          src_ip  flows  n_targets             first_m              last_m  src_wellknown
0   Thursday    192.168.10.8  63251         11 2017-07-06 14:33:00 2017-07-06 15:44:00           0.00
1     Friday   192.168.10.50  17394          1 2017-07-07 14:54:00 2017-07-07 16:16:00           0.98
2  Wednesday   192.168.10.50   2626          1 2017-07-05 09:48:00 2017-07-05 11:00:00           1.00
3  Wednesday  162.208.20.178   2566          1 2017-07-05 15:07:00 2017-07-05 15:21:00           1.00
4   Thursday    142.0.160.53   1719          1 2017-07-06 14:54:00 2017-07-06 15:08:00           1.00
5  Wednesday   162.208.22.34   1658          1 2017-07-05 15:08:00 2017-07-05 15:21:00           1.00
6    Tuesday  173.223.152.83    958          1 2017-07-04 11:07:

In [13]:
# 규칙 개선 → 재실행 → 개선 전후 비교
# 개선 전 성적 보관 (셀을 다시 실행해도 덮어쓰지 않음)
if "KPI_v1" not in globals():
    b = by_label[by_label.label == "BENIGN"].iloc[0]
    KPI_v1 = dict(KPI, fpr=b.alerted / b.flows)

# 개선 1) R1: 서버가 응답하는 방향(출발지 포트 < 1024)은 스캔이 아니므로 제외
RULES[0] = dict(name="R1_PORT_SCAN", keys=["src_ip", "dst_ip"],
    row_filter="f.fwd_pkts <= 3 AND f.src_port >= 1024",
    sql=f'''SELECT src_ip, dst_ip, m, COUNT(*) AS n_flows FROM flows
            WHERE NOT is_baseline AND fwd_pkts <= 3 AND src_port >= 1024
            GROUP BY src_ip, dst_ip, m
            HAVING COUNT(DISTINCT dst_port) >= {SCAN_MIN_PORTS}''')

# 개선 2) R4: 슬로우 DoS는 'TCP 연결을 붙잡는' 공격 → TCP(6)만, 그리고 우리 서버(내부망)를 노린 것만
INTERNAL = "192.168.10.%"
r4_cond = (f"proto = 6 AND dst_ip LIKE '{INTERNAL}' "
           f"AND duration_us >= {SLOW_MIN_DUR_US} AND fwd_bytes <= {SLOW_MAX_FWD_BYTES}")
RULES[3] = dict(name="R4_SLOW_DOS", keys=["src_ip", "dst_ip", "dst_port"],
    row_filter=" AND ".join("f." + c.strip() for c in r4_cond.split(" AND ")),
    sql=f'''SELECT src_ip, dst_ip, dst_port, m, COUNT(*) AS n_flows FROM flows
            WHERE NOT is_baseline AND {r4_cond}
            GROUP BY src_ip, dst_ip, dst_port, m
            HAVING COUNT(*) >= {SLOW_MIN_FLOWS}''')

run_rules()

def evaluate():
    con.execute('''
    CREATE OR REPLACE TABLE flow_eval AS
    SELECT f.fid, f.day, f.label, f.is_attack, f.m, (h.fid IS NOT NULL) AS alerted
    FROM flows f LEFT JOIN (SELECT DISTINCT fid FROM flow_hits) h USING (fid)
    WHERE NOT f.is_baseline''')
    n, att, al, tp, ben, fp = con.sql('''
        SELECT COUNT(*), SUM(is_attack::INT), SUM(alerted::INT), SUM((alerted AND is_attack)::INT),
               SUM((NOT is_attack)::INT), SUM((alerted AND NOT is_attack)::INT)
        FROM flow_eval''').fetchone()
    return dict(alert_rate=al / n, recall=tp / att, precision=tp / al if al else 0.0, fpr=fp / ben)

KPI_v2 = evaluate()
cmp = pd.DataFrame({"개선 전": KPI_v1, "개선 후": KPI_v2}).loc[["alert_rate", "recall", "precision", "fpr"]]
cmp.index = ["경보율", "포착률", "정밀도", "헛경보율(정상 중 경보)"]
print(cmp.map(lambda v: f"{v:.1%}").to_string())

print("\n규칙별 공격/정상 플로우")
print(con.sql('''
    SELECT h.rule,
           COUNT(DISTINCT h.fid) FILTER (WHERE e.is_attack)     AS attack_flows,
           COUNT(DISTINCT h.fid) FILTER (WHERE NOT e.is_attack) AS benign_flows
    FROM flow_hits h JOIN flow_eval e USING (fid)
    GROUP BY h.rule ORDER BY h.rule''').df().to_string())

print("\n공격 유형별 포착률")
by_label = con.sql('''
    SELECT label, COUNT(*) AS flows, SUM(alerted::INT) AS alerted,
           ROUND(AVG(alerted::INT), 4) AS alert_ratio
    FROM flow_eval GROUP BY label, is_attack
    ORDER BY is_attack DESC, flows DESC''').df()
print(by_label.to_string())

                개선 전   개선 후
경보율            29.1%  26.9%
포착률            98.7%  98.7%
정밀도            82.3%  88.9%
헛경보율(정상 중 경보)   6.8%   3.9%

규칙별 공격/정상 플로우
              rule  attack_flows  benign_flows
0     R1_PORT_SCAN        158396         64559
1   R2_BRUTE_FORCE         14476           172
2  R3_VOLUME_FLOOD        375205          4015
3      R4_SLOW_DOS        208697             0

공격 유형별 포착률
                         label    flows   alerted  alert_ratio
0                     DoS Hulk   231073  230850.0       0.9990
1                     PortScan   158930  158481.0       0.9972
2                         DDoS   128027  128024.0       1.0000
3                DoS GoldenEye    10293   10293.0       1.0000
4                  FTP-Patator     7938    7936.0       0.9997
5                  SSH-Patator     5897    5897.0       1.0000
6                DoS slowloris     5796    2349.0       0.4053
7             DoS Slowhttptest     5499    4796.0       0.8722
8                          Bot 

In [ ]:
# 8단계: 라벨 보정 후 재채점

In [14]:
# 라벨 보정 + 3단계 비교
# ① 감염 시간대: Infiltration 라벨의 처음~끝 (데이터에서 직접 계산)
INFECTED = "192.168.10.8"
inf_start, inf_end = con.sql("SELECT MIN(m), MAX(m) FROM flows WHERE label = 'Infiltration'").fetchone()

# ② 보정 대상: 그 시간대에 감염 PC가 '포트 500개 이상'을 두드린 내부 대상과의 플로우 전부
#    (1분 단위 규칙과 별개로, 시간대 전체를 묶어서 판단)
con.execute(f'''
CREATE OR REPLACE TABLE relabel AS
WITH targets AS (
    SELECT dst_ip FROM flows
    WHERE src_ip = '{INFECTED}' AND dst_ip LIKE '192.168.10.%' AND label = 'BENIGN'
      AND m BETWEEN TIMESTAMP '{inf_start}' AND TIMESTAMP '{inf_end}'
    GROUP BY dst_ip HAVING COUNT(DISTINCT dst_port) >= 500)
SELECT f.fid FROM flows f JOIN targets t USING (dst_ip)
WHERE f.src_ip = '{INFECTED}' AND f.label = 'BENIGN'
  AND f.m BETWEEN TIMESTAMP '{inf_start}' AND TIMESTAMP '{inf_end}' ''')

n_relabel, n_relabel_alerted = con.sql('''
    SELECT COUNT(*), SUM(e.alerted::INT) FROM relabel r JOIN flow_eval e USING (fid)''').fetchone()
n_targets = con.sql("SELECT COUNT(DISTINCT f.dst_ip) FROM relabel r JOIN flows f USING (fid)").fetchone()[0]
print(f"감염 시간대: {inf_start} ~ {inf_end}")
print(f"보정 대상: {INFECTED} → 내부 {n_targets}대, {n_relabel:,}건 (그중 경보 {n_relabel_alerted or 0:,}건)\n")

# ③ 보정 라벨로 다시 채점
def evaluate_corrected():
    n, att, al, tp, ben, fp = con.sql('''
        WITH e AS (SELECT e.*, (e.is_attack OR r.fid IS NOT NULL) AS is_attack_c
                   FROM flow_eval e LEFT JOIN relabel r USING (fid))
        SELECT COUNT(*), SUM(is_attack_c::INT), SUM(alerted::INT), SUM((alerted AND is_attack_c)::INT),
               SUM((NOT is_attack_c)::INT), SUM((alerted AND NOT is_attack_c)::INT) FROM e''').fetchone()
    return dict(alert_rate=al / n, recall=tp / att, precision=tp / al if al else 0.0, fpr=fp / ben)

KPI_v3 = evaluate_corrected()
cmp = pd.DataFrame({"개선 전": KPI_v1, "개선 후": KPI_v2, "개선 후 + 라벨 보정": KPI_v3}
                   ).loc[["alert_rate", "recall", "precision", "fpr"]]
cmp.index = ["경보율", "포착률", "정밀도", "헛경보율(정상 중 경보)"]
print(cmp.map(lambda v: f"{v:.1%}").to_string())

감염 시간대: 2017-07-06 14:19:00 ~ 2017-07-06 15:45:00
보정 대상: 192.168.10.8 → 내부 11대, 64,741건 (그중 경보 63,313건)

                개선 전   개선 후 개선 후 + 라벨 보정
경보율            29.1%  26.9%        26.9%
포착률            98.7%  98.7%        98.6%
정밀도            82.3%  88.9%        99.1%
헛경보율(정상 중 경보)   6.8%   3.9%         0.3%


In [ ]:
# 9단계: 경보를 인시던트로 묶기

In [15]:
# 보정 라벨 저장 + 인시던트로 묶기

# 보정 라벨 테이블: 이후 단계에서는 보정된 라벨로 평가
con.execute('''
CREATE OR REPLACE TABLE label_c AS
SELECT f.fid,
       CASE WHEN r.fid IS NOT NULL THEN 'Infiltration - Internal Recon' ELSE f.label END AS label_c,
       (f.is_attack OR r.fid IS NOT NULL) AS is_attack_c
FROM flows f LEFT JOIN relabel r USING (fid)''')

# Gaps & Islands: 같은 규칙·같은 대상의 경보가 2분 이내로 이어지면 하나의 인시던트
con.execute('''
CREATE OR REPLACE TABLE incidents_raw AS
WITH g AS (                                   -- ① 경보마다 '대상' 이름표 만들기
    SELECT *, concat_ws('|', rule, COALESCE(src_ip, '*'), COALESCE(dst_ip, '*'),
                        COALESCE(CAST(dst_port AS VARCHAR), '*')) AS entity
    FROM alert_groups),
s AS (                                        -- ② 직전 경보와 2분 넘게 떨어지면 '새 시작'(1)
    SELECT *, CASE WHEN lag(m) OVER w IS NULL OR m - lag(m) OVER w > INTERVAL 2 MINUTE
                   THEN 1 ELSE 0 END AS is_new
    FROM g WINDOW w AS (PARTITION BY entity ORDER BY m)),
t AS (                                        -- ③ '새 시작'을 누적합 → 같은 숫자 = 같은 섬
    SELECT *, SUM(is_new) OVER (PARTITION BY entity ORDER BY m ROWS UNBOUNDED PRECEDING) AS seq
    FROM s)
SELECT gid, entity, rule, m, dense_rank() OVER (ORDER BY entity, seq) AS inc_id FROM t''')

n_alerts = con.sql("SELECT COUNT(*) FROM alert_groups").fetchone()[0]
n_inc = con.sql("SELECT COUNT(DISTINCT inc_id) FROM incidents_raw").fetchone()[0]
print(f"분 단위 경보 {n_alerts:,}건 → 인시던트 {n_inc:,}건 ({n_alerts / n_inc:.1f}배 압축)")

분 단위 경보 477건 → 인시던트 105건 (4.5배 압축)


In [16]:
# 인시던트별 행위 요약 만들기

def top_k(col, k):
    return f'''
    WITH c AS (SELECT i.inc_id, CAST(f.{col} AS VARCHAR) AS v, COUNT(*) AS n
               FROM incidents_raw i JOIN flow_hits h USING (gid) JOIN flows f USING (fid) GROUP BY ALL),
         r AS (SELECT *, row_number() OVER (PARTITION BY inc_id ORDER BY n DESC) AS rn FROM c)
    SELECT inc_id, string_agg(v || ' (' || n || ')', ', ' ORDER BY n DESC) AS top_{col}
    FROM r WHERE rn <= {k} GROUP BY inc_id'''

con.execute(f'''
CREATE OR REPLACE TABLE incidents AS
WITH base AS (
    SELECT i.inc_id, any_value(i.rule) AS rule, any_value(i.entity) AS entity,
           MIN(f.ts) AS start_ts, MAX(f.ts) AS end_ts, COUNT(DISTINCT f.m) AS active_minutes,
           COUNT(*) AS n_flows, COUNT(DISTINCT f.src_ip) AS n_src_ips, COUNT(DISTINCT f.dst_ip) AS n_dst_ips,
           COUNT(DISTINCT f.dst_port) AS n_dst_ports, COUNT(DISTINCT f.src_port) AS n_src_ports,
           ROUND(AVG(f.duration_us) / 1000, 1) AS avg_flow_ms,
           ROUND(quantile_cont(f.duration_us, 0.5) / 1000, 1) AS median_flow_ms,
           ROUND(AVG(f.fwd_pkts), 2) AS avg_fwd_pkts, ROUND(AVG(f.bwd_pkts), 2) AS avg_bwd_pkts,
           ROUND(AVG(f.fwd_bytes), 1) AS avg_fwd_bytes, ROUND(AVG(f.bwd_bytes), 1) AS avg_bwd_bytes,
           ROUND(AVG((f.bwd_pkts = 0)::INT), 3) AS no_response_ratio,
           ROUND(AVG((f.syn > 0)::INT), 3) AS syn_ratio, ROUND(AVG((f.rst > 0)::INT), 3) AS rst_ratio,
           mode(f.proto) AS main_proto,
           -- ↓ 평가 전용 (LLM에는 절대 넘기지 않음)
           mode(l.label_c) AS top_label, ROUND(AVG(l.is_attack_c::INT), 3) AS attack_share
    FROM incidents_raw i JOIN flow_hits h USING (gid)
         JOIN flows f USING (fid) JOIN label_c l USING (fid)
    GROUP BY i.inc_id)
SELECT base.*, p.top_dst_port, s.top_src_ip, d.top_dst_ip
FROM base
LEFT JOIN ({top_k("dst_port", 5)}) p USING (inc_id)
LEFT JOIN ({top_k("src_ip", 3)}) s USING (inc_id)
LEFT JOIN ({top_k("dst_ip", 3)}) d USING (inc_id)''')

print("규칙별 인시던트")
print(con.sql('''
    SELECT rule, COUNT(*) AS incidents,
           SUM((attack_share >= 0.5)::INT) AS attack_incidents,
           SUM((attack_share <  0.5)::INT) AS false_positive_incidents,
           SUM(n_flows) AS flows
    FROM incidents GROUP BY rule ORDER BY rule''').df().to_string())

print("\n규모 상위 15개 인시던트")
print(con.sql('''
    SELECT inc_id, rule, start_ts, active_minutes AS mins, n_flows, n_dst_ports,
           top_label, attack_share
    FROM incidents ORDER BY n_flows DESC LIMIT 15''').df().to_string())

print("\n헛경보 인시던트 (공격 비율 50% 미만) 상위 10개")
print(con.sql('''
    SELECT inc_id, rule, start_ts, n_flows, top_dst_port, top_src_ip, top_dst_ip, attack_share
    FROM incidents WHERE attack_share < 0.5 ORDER BY n_flows DESC LIMIT 10''').df().to_string())

규칙별 인시던트
              rule  incidents  attack_incidents  false_positive_incidents     flows
0     R1_PORT_SCAN         52              50.0                       2.0  222955.0
1   R2_BRUTE_FORCE         22              22.0                       0.0   14648.0
2  R3_VOLUME_FLOOD         18              12.0                       6.0  379220.0
3      R4_SLOW_DOS         13              13.0                       0.0  208697.0

규모 상위 15개 인시던트
    inc_id             rule            start_ts  mins  n_flows  n_dst_ports                      top_label  attack_share
0       88  R3_VOLUME_FLOOD 2017-07-05 10:43:00    18   230726            1                       DoS Hulk           1.0
1        1     R1_PORT_SCAN 2017-07-07 14:51:00     6   138490          997                       PortScan           1.0
2      100      R4_SLOW_DOS 2017-07-05 10:43:00    18   136824            1                       DoS Hulk           1.0
3       91  R3_VOLUME_FLOOD 2017-07-07 15:56:00    21   128024         

In [ ]:
# 10단계: Claude로 ATT&CK 트리아지

In [17]:
# MITRE ATT&CK 공식 데이터 불러오기 + 채점표

import requests
stix = requests.get("https://raw.githubusercontent.com/mitre/cti/master/enterprise-attack/enterprise-attack.json",
                    timeout=180).json()
ATTACK = {}
for obj in stix["objects"]:
    if obj.get("type") != "attack-pattern":
        continue
    ref = next((r for r in obj.get("external_references", []) if r.get("source_name") == "mitre-attack"), None)
    if ref:
        ATTACK[ref["external_id"]] = dict(
            name=obj["name"],
            tactics=[p["phase_name"] for p in obj.get("kill_chain_phases", [])],
            retired=bool(obj.get("revoked") or obj.get("x_mitre_deprecated")))
print(f"ATT&CK Enterprise 기법 {len(ATTACK):,}개 로드")
print("예시:", "T1046 →", ATTACK["T1046"]["name"], "/", ATTACK["T1110.001"]["name"])

# 채점용 정답표: 데이터셋 라벨 → 허용하는 ATT&CK 부모 기법
TRUTH = {
    "portscan": {"T1046"}, "infiltration - internal recon": {"T1046"},
    "ftp-patator": {"T1110"}, "ssh-patator": {"T1110"},
    "dos hulk": {"T1499", "T1498"}, "dos goldeneye": {"T1499", "T1498"},
    "dos slowloris": {"T1499"}, "dos slowhttptest": {"T1499"}, "ddos": {"T1498", "T1499"},
    "web attack - brute force": {"T1110"}, "web attack - xss": {"T1059", "T1189", "T1190"},
    "web attack - sql injection": {"T1190"}, "heartbleed": {"T1190"},
    "infiltration": {"T1046", "T1204", "T1105"}, "bot": {"T1071", "T1571", "T1095"},
}
# 비교 기준선: 규칙 이름만 보고 기법을 고정 매핑했을 때
RULE_BASELINE = {"R1_PORT_SCAN": "T1046", "R2_BRUTE_FORCE": "T1110",
                 "R3_VOLUME_FLOOD": "T1498", "R4_SLOW_DOS": "T1499"}

ATT&CK Enterprise 기법 858개 로드
예시: T1046 → Network Service Discovery / Password Guessing


In [18]:
# Claude에게 보낼 입력 만들기

import ipaddress
API_KEY = None
try:
    from google.colab import userdata
    API_KEY = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    API_KEY = os.environ.get("ANTHROPIC_API_KEY")
print("API 키:", "확인됨" if API_KEY else "없음 → 코랩 왼쪽 🔑에 ANTHROPIC_API_KEY 등록 필요")

# IP 가명 처리: 실제 IP 대신 '내부/외부' 구분만 남김 (공개 데이터셋 IP를 알아보고 답을 맞히는 것 방지)
IP_ALIAS = {}
def alias(ip):
    if ip not in IP_ALIAS:
        kind = "int" if ipaddress.ip_address(ip).is_private else "ext"
        IP_ALIAS[ip] = f"{kind}-{sum(v.startswith(kind) for v in IP_ALIAS.values()) + 1:02d}"
    return IP_ALIAS[ip]
def mask_ips(text):
    return re.sub(r"\b\d{1,3}(?:\.\d{1,3}){3}\b", lambda m: alias(m.group(0)), str(text))

FEATURES = ["active_minutes", "n_flows", "n_src_ips", "n_dst_ips", "n_dst_ports", "n_src_ports",
            "avg_flow_ms", "median_flow_ms", "avg_fwd_pkts", "avg_bwd_pkts", "avg_fwd_bytes", "avg_bwd_bytes",
            "no_response_ratio", "syn_ratio", "rst_ratio", "main_proto", "top_dst_port", "top_src_ip", "top_dst_ip"]

def build_payload(row):
    p = {}
    for k in FEATURES:
        v = row[k]
        if isinstance(v, (np.integer, np.floating)):
            v = None if pd.isna(v) else v.item()
        p[k] = mask_ips(v) if k in ("top_src_ip", "top_dst_ip") else v
    p["duration_minutes"] = round((row.end_ts - row.start_ts).total_seconds() / 60, 1)
    return p

SYSTEM = '''You are a Tier-1 SOC analyst triaging a network incident produced by an automated detection pipeline.
You only see aggregated flow statistics for the incident (no payloads, no detection rule name).
"top_*" fields are "value (flow count)". IPs are pseudonymized: int-XX = private/internal address, ext-XX = public internet.
main_proto: 6=TCP, 17=UDP. Durations are in milliseconds unless stated.
Decide whether the activity is malicious and map it to the single most likely MITRE ATT&CK Enterprise technique.
Rules:
- Use only real ATT&CK Enterprise technique IDs (e.g. T1046, T1110.001). If benign, technique_id must be null.
- Before calling it malicious, actively consider benign explanations (busy servers, browsers opening many
  connections to a CDN, server responses recorded in reverse direction, backups, monitoring).
- Cite concrete numbers from the input as evidence.
Return ONLY a JSON object with keys:
verdict ("malicious"|"suspicious"|"benign"), technique_id, technique_name, tactic,
confidence (0-1), severity ("low"|"medium"|"high"|"critical"),
evidence (list of short strings), recommended_actions (list of short strings),
summary_ko (two sentences in Korean).'''

incidents = con.sql("SELECT * FROM incidents ORDER BY inc_id").df()
print(f"트리아지 대상 인시던트 {len(incidents)}건\n")
print("▼ Claude가 실제로 받는 입력 예시 (규모 1위 인시던트)")
print(json.dumps(build_payload(incidents.loc[incidents.n_flows.idxmax()]), ensure_ascii=False, indent=2))

API 키: 확인됨
트리아지 대상 인시던트 105건

▼ Claude가 실제로 받는 입력 예시 (규모 1위 인시던트)
{
  "active_minutes": 18,
  "n_flows": 230726,
  "n_src_ips": 1,
  "n_dst_ips": 1,
  "n_dst_ports": 1,
  "n_src_ports": 14108,
  "avg_flow_ms": 57150.6,
  "median_flow_ms": 84934.3,
  "avg_fwd_pkts": 5.28,
  "avg_bwd_pkts": 4.21,
  "avg_fwd_bytes": 281.8,
  "avg_bwd_bytes": 7782.4,
  "no_response_ratio": 0.29,
  "syn_ratio": 0.0,
  "rst_ratio": 0.0,
  "main_proto": 6,
  "top_dst_port": "80 (230726)",
  "top_src_ip": "int-01 (230726)",
  "top_dst_ip": "int-02 (230726)",
  "duration_minutes": 17.0
}


In [19]:
# [1차 시도 기록] max_tokens=1200 → 답이 잘려 68건 JSON 파싱 실패. 다음 셀에서 4000으로 늘려 해결

# import anthropic, time
# client = anthropic.Anthropic(api_key=API_KEY, max_retries=5)

# CACHE_PATH = f"{OUT_DIR}/llm_cache.json"
# cache = json.load(open(CACHE_PATH)) if os.path.exists(CACHE_PATH) else {}

# def triage(row):
#     key = f"{ANTHROPIC_MODEL}|{row.entity}|{row.start_ts}"
#     if key in cache:                                   # 이미 물어본 건 다시 묻지 않음 (비용 절약)
#         return cache[key]
#     payload = json.dumps(build_payload(row), ensure_ascii=False)
#     for attempt in range(3):
#         resp = client.messages.create(model=ANTHROPIC_MODEL, max_tokens=1200, system=SYSTEM,
#                                       messages=[{"role": "user", "content": payload}])
#         text = "".join(b.text for b in resp.content if b.type == "text")
#         m = re.search(r"\{.*\}", text, re.S)
#         try:
#             out = json.loads(m.group(0))
#             cache[key] = out
#             json.dump(cache, open(CACHE_PATH, "w"), ensure_ascii=False)
#             return out
#         except Exception:
#             time.sleep(1)
#     return {"verdict": "error"}

# results = []
# for i, row in incidents.iterrows():
#     results.append(triage(row))
#     print(f"\r{i + 1}/{len(incidents)}", end="")
# print()

# triaged = pd.concat([incidents.reset_index(drop=True), pd.json_normalize(results).add_prefix("llm_")], axis=1)
# triaged.to_csv(f"{OUT_DIR}/triaged_incidents.csv", index=False)
# print(triaged.llm_verdict.value_counts().to_string())

105/105
llm_verdict
error         68
malicious     20
benign        10
suspicious     7


In [21]:
# 105건 트리아지 실행(v1)
import anthropic, time
client = anthropic.Anthropic(api_key=API_KEY, max_retries=5)
CACHE_PATH = f"{OUT_DIR}/llm_cache.json"
cache = json.load(open(CACHE_PATH)) if os.path.exists(CACHE_PATH) else {}


MAX_TOKENS = 4000          # 1200 → 4000: 답이 길면 중간에 잘려서 JSON이 깨졌던 것으로 추정
failures = {}              # 실패한 응답의 원인 기록

def parse_json(text):
    start = text.find("{")
    if start < 0:
        return None
    try:
        return json.JSONDecoder().raw_decode(text[start:])[0]   # 첫 JSON 객체만 정확히 읽기
    except json.JSONDecodeError:
        return None

def triage(row):
    key = f"{ANTHROPIC_MODEL}|{row.entity}|{row.start_ts}"
    if key in cache:                                   # 성공한 답은 캐시에서 재사용 (비용 X)
        return cache[key]
    payload = json.dumps(build_payload(row), ensure_ascii=False)
    for attempt in range(3):
        resp = client.messages.create(model=ANTHROPIC_MODEL, max_tokens=MAX_TOKENS, system=SYSTEM,
                                      messages=[{"role": "user", "content": payload}])
        text = "".join(b.text for b in resp.content if b.type == "text")
        out = parse_json(text)
        if isinstance(out, dict) and "verdict" in out:
            cache[key] = out
            json.dump(cache, open(CACHE_PATH, "w"), ensure_ascii=False)
            return out
        failures[row.inc_id] = dict(stop_reason=resp.stop_reason, n_chars=len(text), tail=text[-300:])
        time.sleep(1)
    return {"verdict": "error"}

results = []
for i, row in incidents.iterrows():
    results.append(triage(row))
    print(f"\r{i + 1}/{len(incidents)}", end="")
print()

triaged = pd.concat([incidents.reset_index(drop=True), pd.json_normalize(results).add_prefix("llm_")], axis=1)
triaged.to_csv(f"{OUT_DIR}/triaged_incidents.csv", index=False)
print(triaged.llm_verdict.value_counts().to_string())

if failures:
    print(f"\n여전히 실패: {len(failures)}건")
    print(pd.Series([f["stop_reason"] for f in failures.values()]).value_counts().to_string())
    first = next(iter(failures.items()))
    print(f"\n예시 (inc_id={first[0]}) 응답 끝부분:\n{first[1]['tail']}")

105/105
llm_verdict
malicious     46
suspicious    43
benign        16


In [22]:
# Claude의 답 채점
def parent(tid):
    return str(tid).split(".")[0] if isinstance(tid, str) and tid else None

t = triaged.copy()
t["is_attack_inc"] = t.attack_share >= 0.5
t["truth"] = t.top_label.str.lower().map(TRUTH)
t["id_valid"] = t.llm_technique_id.map(lambda x: isinstance(x, str) and x in ATTACK and not ATTACK[x]["retired"])
t["llm_match"] = [parent(x) in s if isinstance(s, set) else np.nan for x, s in zip(t.llm_technique_id, t.truth)]
t["rule_match"] = [RULE_BASELINE[r] in s if isinstance(s, set) else np.nan for r, s in zip(t.rule, t.truth)]

att, fp = t[t.is_attack_inc], t[~t.is_attack_inc]
named = t[t.llm_technique_id.map(lambda x: isinstance(x, str) and x != "")]
LLM_KPI = {
    "트리아지 인시던트":                 len(t),
    "기법 ID 유효율 (실존·미폐기)":       f"{named.id_valid.mean():.0%}" if len(named) else "-",
    "공격 인시던트":                    len(att),
    "  └ 악성/의심으로 판정":            f"{(att.llm_verdict != 'benign').mean():.0%}",
    "  └ ATT&CK 매핑 정확도 (LLM)":      f"{att.llm_match.astype(float).mean():.0%}",
    "  └ ATT&CK 매핑 정확도 (규칙명 고정)": f"{att.rule_match.astype(float).mean():.0%}",
    "헛경보 인시던트":                   len(fp),
    "  └ LLM이 정상으로 걸러냄":          f"{(fp.llm_verdict == 'benign').mean():.0%}" if len(fp) else "-",
}
print(pd.Series(LLM_KPI).to_string())
print("\n존재하지 않거나 폐기된 기법 ID:", named.loc[~named.id_valid, "llm_technique_id"].tolist())

print("\n공격 유형별 매핑 정확도")
print(t[t.is_attack_inc].groupby("top_label").agg(
    incidents=("inc_id", "count"), llm_acc=("llm_match", "mean"), rule_acc=("rule_match", "mean"),
    llm_ids=("llm_technique_id", lambda s: ", ".join(sorted(set(map(str, s)))))).to_string())

print("\n헛경보 인시던트에 대한 LLM 판정")
print(fp[["inc_id", "rule", "n_flows", "top_dst_port", "llm_verdict", "llm_technique_id",
          "llm_confidence", "llm_summary_ko"]].to_string())

트리아지 인시던트                      105
기법 ID 유효율 (실존·미폐기)            100%
공격 인시던트                         97
  └ 악성/의심으로 판정                 90%
  └ ATT&CK 매핑 정확도 (LLM)        74%
  └ ATT&CK 매핑 정확도 (규칙명 고정)     68%
헛경보 인시던트                         8
  └ LLM이 정상으로 걸러냄              75%

존재하지 않거나 폐기된 기법 ID: []

공격 유형별 매핑 정확도
                               incidents   llm_acc  rule_acc                                              llm_ids
top_label                                                                                                        
DDoS                                   2       1.0       1.0                                            T1499.002
DoS GoldenEye                          4      0.75       1.0                                      None, T1499.002
DoS Hulk                               2       0.5       1.0                                      None, T1499.002
DoS Slowhttptest                      10       1.0       0.5                                            T1499.0

In [ ]:
# 11단계: AI 판정을 실제로 어떻게 쓸 수 있을까

In [23]:
# 놓친 공격 확인 + 자동 종결 정책 시뮬레이션

# ① Claude가 '정상'이라고 놓친 공격 인시던트
miss = t[t.is_attack_inc & (t.llm_verdict == "benign")]
print(f"공격인데 정상으로 판정: {len(miss)}건\n")
print(miss[["inc_id", "rule", "top_label", "n_flows", "n_dst_ports", "top_dst_port",
            "llm_confidence", "llm_summary_ko"]].to_string())

# ② 운영 정책 시뮬레이션: "정상 + 확신도 c 이상"이면 자동 종결한다면?
rows = []
for c in [0.5, 0.6, 0.7, 0.8, 0.9]:
    closed = t[(t.llm_verdict == "benign") & (t.llm_confidence.astype(float) >= c)]
    rows.append({"확신도 기준": f"≥ {c}",
                 "자동 종결": len(closed),
                 "└ 헛경보를 바르게 종결": int((~closed.is_attack_inc).sum()),
                 "└ 공격을 잘못 종결": int(closed.is_attack_inc.sum()),
                 "사람이 볼 인시던트": len(t) - len(closed)})
print("\n자동 종결 정책 시뮬레이션")
print(pd.DataFrame(rows).to_string(index=False))

공격인데 정상으로 판정: 10건

     inc_id             rule                      top_label  n_flows  n_dst_ports                                       top_dst_port  llm_confidence                                                                                                                                                                                                                                                               llm_summary_ko
2         3     R1_PORT_SCAN                       PortScan     8954          998  80 (25), 2000 (17), 27000 (9), 1583 (9), 1580 (9)            0.60  내부 호스트 한 대(int-01)가 내부 서버 한 대(int-02)와 약 1분간 8,954개의 짧은 플로우를 주고받았고, 모든 플로우에 응답이 있으며 SYN/RST 비율이 0이라 전형적인 포트 스캔 패턴과 맞지 않습니다. 목적지 포트가 998개로 많아 보이지만 출발지 포트 42개와 포트당 9~25개 수준의 분포로 볼 때 서버 응답이 역방향으로 기록되었거나 라이선스/DB 폴링 트래픽일 가능성이 높아 양성으로 판단하되, 해당 호스트 쌍이 알려진 정상 통신인지 확인이 필요합니다.
57       58   R2_BRUTE_FORCE                       PortScan       10            1                                            21 (10)            0

In [24]:
# 규칙 간 상관분석: 인시던트 → 사건 (v2)

# v2-1. 규칙 간 상관분석: 같은 목적지 + 시간 겹침 + (출발지 같거나 규칙이 출발지를 안 봄) → 하나의 사건(case)
CORR_GAP_MIN = 2   # 앞뒤 2분 여유

con.execute('''
CREATE OR REPLACE TABLE inc_span AS
SELECT i.inc_id, any_value(i.rule) AS rule, any_value(g.src_ip) AS src_ip, any_value(g.dst_ip) AS dst_ip,
       MIN(i.m) AS start_m, MAX(i.m) AS end_m
FROM incidents_raw i JOIN alert_groups g USING (gid)
GROUP BY i.inc_id''')

pairs = con.sql(f'''
    SELECT a.inc_id AS a, b.inc_id AS b
    FROM inc_span a JOIN inc_span b
      ON a.inc_id < b.inc_id
     AND a.dst_ip = b.dst_ip
     AND (a.src_ip = b.src_ip OR a.src_ip IS NULL OR b.src_ip IS NULL)
     AND a.start_m <= b.end_m + INTERVAL {CORR_GAP_MIN} MINUTE
     AND b.start_m <= a.end_m + INTERVAL {CORR_GAP_MIN} MINUTE''').fetchall()

# Union-Find: 겹치는 인시던트끼리 같은 그룹으로 묶기 (A-B, B-C가 겹치면 A-B-C 한 그룹)
parent_of = {i: i for i in con.sql("SELECT inc_id FROM inc_span").df().inc_id}
def find(x):
    while parent_of[x] != x:
        parent_of[x] = parent_of[parent_of[x]]
        x = parent_of[x]
    return x
for a, b in pairs:
    parent_of[find(a)] = find(b)

inc_case = pd.DataFrame({"inc_id": list(parent_of)})
inc_case["root"] = inc_case.inc_id.map(find)
inc_case["case_id"] = inc_case.root.rank(method="dense").astype(int)
con.register("inc_case_df", inc_case[["inc_id", "case_id"]])
con.execute("CREATE OR REPLACE TABLE inc_case AS SELECT * FROM inc_case_df")

# 사건 단위 플로우 (여러 규칙에 걸린 같은 플로우는 한 번만)
con.execute('''
CREATE OR REPLACE TABLE case_flows AS
SELECT DISTINCT c.case_id, h.fid
FROM incidents_raw i JOIN inc_case c USING (inc_id) JOIN flow_hits h USING (gid)''')

n_inc, n_case = len(inc_case), inc_case.case_id.nunique()
print(f"인시던트 {n_inc}건 → 사건 {n_case}건 (규칙 간 상관분석으로 {n_inc - n_case}건 병합)\n")
print("여러 인시던트가 합쳐진 사건")
print(con.sql('''
    SELECT c.case_id, COUNT(*) AS n_incidents,
           string_agg(DISTINCT s.rule, ', ' ORDER BY s.rule) AS rules,
           MIN(s.start_m) AS start_m, MAX(s.end_m) AS end_m
    FROM inc_case c JOIN inc_span s USING (inc_id)
    GROUP BY c.case_id HAVING COUNT(*) > 1
    ORDER BY n_incidents DESC''').df().to_string())

인시던트 105건 → 사건 74건 (규칙 간 상관분석으로 31건 병합)

여러 인시던트가 합쳐진 사건
    case_id  n_incidents                         rules             start_m               end_m
0        51            5  R1_PORT_SCAN, R2_BRUTE_FORCE 2017-07-07 14:51:00 2017-07-07 14:56:00
1        63            4  R3_VOLUME_FLOOD, R4_SLOW_DOS 2017-07-05 09:48:00 2017-07-05 10:09:00
2        49            3  R1_PORT_SCAN, R2_BRUTE_FORCE 2017-07-07 15:08:00 2017-07-07 15:10:00
3        52            3  R1_PORT_SCAN, R2_BRUTE_FORCE 2017-07-06 15:13:00 2017-07-06 15:13:00
4        54            3  R1_PORT_SCAN, R2_BRUTE_FORCE 2017-07-06 15:32:00 2017-07-06 15:32:00
5        50            3  R1_PORT_SCAN, R2_BRUTE_FORCE 2017-07-07 15:22:00 2017-07-07 15:23:00
6        53            3  R1_PORT_SCAN, R2_BRUTE_FORCE 2017-07-06 15:23:00 2017-07-06 15:23:00
7        55            3  R1_PORT_SCAN, R2_BRUTE_FORCE 2017-07-06 15:41:00 2017-07-06 15:41:00
8        62            2   R2_BRUTE_FORCE, R4_SLOW_DOS 2017-07-04 09:19:00 2017-07-04 10

In [25]:
# 사건별 특성 + 맥락 보강

# v2-2. 사건별 특성 + 맥락 보강(enrichment)

# (a) 정상 기준선(월요일): 출발지→목적지 쌍이 평소에도 있었나 / 출발지가 평소 분당 몇 개 포트를 건드렸나
con.execute('''
CREATE OR REPLACE TABLE base_pairs AS
SELECT DISTINCT src_ip, dst_ip FROM flows WHERE is_baseline''')
con.execute('''
CREATE OR REPLACE TABLE base_src_ports AS
SELECT src_ip, MAX(c) AS max_ports_per_min FROM (
    SELECT src_ip, m, COUNT(DISTINCT dst_port) AS c FROM flows WHERE is_baseline GROUP BY src_ip, m)
GROUP BY src_ip''')

def top_k_case(col, k):
    return f'''
    WITH c AS (SELECT cf.case_id, CAST(f.{col} AS VARCHAR) AS v, COUNT(*) AS n
               FROM case_flows cf JOIN flows f USING (fid) GROUP BY ALL),
         r AS (SELECT *, row_number() OVER (PARTITION BY case_id ORDER BY n DESC, v) AS rn FROM c)
    SELECT case_id, string_agg(v || ' (' || n || ')', ', ' ORDER BY n DESC, v) AS top_{col}
    FROM r WHERE rn <= {k} GROUP BY case_id'''

con.execute(f'''
CREATE OR REPLACE TABLE cases AS
WITH base AS (
    SELECT cf.case_id,
           MIN(f.ts) AS start_ts, MAX(f.ts) AS end_ts, COUNT(DISTINCT f.m) AS active_minutes,
           COUNT(*) AS n_flows, COUNT(DISTINCT f.src_ip) AS n_src_ips, COUNT(DISTINCT f.dst_ip) AS n_dst_ips,
           COUNT(DISTINCT f.dst_port) AS n_dst_ports, COUNT(DISTINCT f.src_port) AS n_src_ports,
           ROUND(AVG(f.duration_us) / 1000, 1) AS avg_flow_ms,
           ROUND(quantile_cont(f.duration_us, 0.5) / 1000, 1) AS median_flow_ms,
           ROUND(AVG(f.fwd_pkts), 2) AS avg_fwd_pkts, ROUND(AVG(f.bwd_pkts), 2) AS avg_bwd_pkts,
           ROUND(AVG(f.fwd_bytes), 1) AS avg_fwd_bytes, ROUND(AVG(f.bwd_bytes), 1) AS avg_bwd_bytes,
           ROUND(AVG((f.bwd_pkts = 0)::INT), 3) AS no_response_ratio,
           mode(f.proto) AS main_proto,
           mode(f.src_ip) AS main_src, mode(f.dst_ip) AS main_dst, mode(f.dst_port) AS main_dport,
           -- ↓ 평가 전용 (LLM에는 넘기지 않음)
           mode(l.label_c) AS top_label, ROUND(AVG(l.is_attack_c::INT), 3) AS attack_share,
           SUM(l.is_attack_c::INT) AS attack_flows
    FROM case_flows cf JOIN flows f USING (fid) JOIN label_c l USING (fid)
    GROUP BY cf.case_id),
peak AS (
    SELECT cf.case_id, MAX(c) AS peak_flows_per_min FROM (
        SELECT cf.case_id, f.m, COUNT(*) AS c FROM case_flows cf JOIN flows f USING (fid) GROUP BY ALL) cf
    GROUP BY cf.case_id),
rules AS (   -- 평가 전용: 사건 안에서 플로우가 가장 많은 규칙 = 주 규칙
    SELECT c.case_id, arg_max(i.rule, i.n_flows) AS primary_rule,
           string_agg(DISTINCT i.rule, ', ' ORDER BY i.rule) AS rules, COUNT(*) AS n_incidents
    FROM inc_case c JOIN incidents i USING (inc_id) GROUP BY c.case_id)
SELECT base.*, peak.peak_flows_per_min, rules.primary_rule, rules.rules, rules.n_incidents,
       p.top_dst_port, s.top_src_ip, d.top_dst_ip,
       -- 맥락 보강: 평소(월요일)와 비교
       (fb.p99 IS NOT NULL)                AS service_seen_in_baseline,
       COALESCE(fb.p99, 0)                 AS service_baseline_p99_flows_per_min,
       (bp.src_ip IS NOT NULL)             AS pair_seen_in_baseline,
       COALESCE(bs.max_ports_per_min, 0)   AS src_baseline_max_ports_per_min
FROM base JOIN peak USING (case_id) JOIN rules USING (case_id)
LEFT JOIN ({top_k_case("dst_port", 5)}) p USING (case_id)
LEFT JOIN ({top_k_case("src_ip", 3)}) s USING (case_id)
LEFT JOIN ({top_k_case("dst_ip", 3)}) d USING (case_id)
LEFT JOIN flood_baseline fb ON fb.dst_ip = base.main_dst AND fb.dst_port = base.main_dport
LEFT JOIN base_pairs bp ON bp.src_ip = base.main_src AND bp.dst_ip = base.main_dst
LEFT JOIN base_src_ports bs ON bs.src_ip = base.main_src
ORDER BY case_id''')

# (b) 자산 정보: 데이터셋 공개 네트워크 구성도 기준 (회사의 자산관리대장(CMDB)에 해당)
ASSET_ROLES = {
    "172.16.0.1":    "firewall internal interface; inbound internet traffic is NATed to this address",
    "192.168.10.3":  "domain controller / DNS server",
    "192.168.10.50": "web server",
    "192.168.10.51": "Linux server",
}
def mask_ips_v2(text):
    def rep(m):
        ip = m.group(0)
        role = ASSET_ROLES.get(ip)
        return f"{alias(ip)} [{role}]" if role else alias(ip)
    return re.sub(r"\b\d{1,3}(?:\.\d{1,3}){3}\b", rep, str(text))

FEATURES_V2 = ["active_minutes", "n_flows", "peak_flows_per_min", "n_src_ips", "n_dst_ips", "n_dst_ports",
               "n_src_ports", "avg_flow_ms", "median_flow_ms", "avg_fwd_pkts", "avg_bwd_pkts",
               "avg_fwd_bytes", "avg_bwd_bytes", "no_response_ratio", "main_proto",
               "top_dst_port", "top_src_ip", "top_dst_ip",
               "service_seen_in_baseline", "service_baseline_p99_flows_per_min",
               "pair_seen_in_baseline", "src_baseline_max_ports_per_min"]   # syn/rst 비율은 제외 (집계 오류)

def build_payload_v2(row):
    p = {}
    for k in FEATURES_V2:
        v = row[k]
        if isinstance(v, (np.integer, np.floating, np.bool_)):
            v = None if pd.isna(v) else v.item()
        p[k] = mask_ips_v2(v) if k in ("top_src_ip", "top_dst_ip") else v
    p["duration_minutes"] = round((row.end_ts - row.start_ts).total_seconds() / 60, 1)
    return p

SYSTEM_V2 = SYSTEM.replace(
    "main_proto: 6=TCP, 17=UDP.",
    "main_proto: 6=TCP, 17=UDP. Known asset roles from the asset inventory appear in [brackets] after an IP.\n"
    "Baseline context comes from a known-clean day of the same network: "
    "service_seen_in_baseline / service_baseline_p99_flows_per_min describe normal load on the main destination "
    "service; pair_seen_in_baseline says whether the main source ever talked to the main destination on that day; "
    "src_baseline_max_ports_per_min is the most distinct destination ports the main source touched in any minute "
    "on that day (0 = source not seen). TCP flag counts are unavailable.")

cases = con.sql("SELECT * FROM cases ORDER BY case_id").df()
print(f"사건 {len(cases)}건 준비\n")
for title, idx in [("PortScan 계열 최대 사건", cases[cases.top_label == "PortScan"].n_flows.idxmax()),
                   ("규모 1위 사건", cases.n_flows.idxmax())]:
    print(f"▼ {title} (case_id={cases.loc[idx, 'case_id']}) — Claude가 받는 입력")
    print(json.dumps(build_payload_v2(cases.loc[idx]), ensure_ascii=False, indent=2), "\n")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

사건 74건 준비

▼ PortScan 계열 최대 사건 (case_id=51) — Claude가 받는 입력
{
  "active_minutes": 6,
  "n_flows": 138490,
  "peak_flows_per_min": 44410,
  "n_src_ips": 1,
  "n_dst_ips": 1,
  "n_dst_ports": 997,
  "n_src_ports": 14094,
  "avg_flow_ms": 88.0,
  "median_flow_ms": 0.0,
  "avg_fwd_pkts": 1.01,
  "avg_bwd_pkts": 1.0,
  "avg_fwd_bytes": 0.9,
  "avg_bwd_bytes": 6.0,
  "no_response_ratio": 0.0,
  "main_proto": 6,
  "top_dst_port": "80 (143), 1 (140), 1000 (140), 10000 (140), 10001 (140)",
  "top_src_ip": "int-01 [firewall internal interface; inbound internet traffic is NATed to this address] (138490)",
  "top_dst_ip": "int-02 [web server] (138490)",
  "service_seen_in_baseline": false,
  "service_baseline_p99_flows_per_min": 0.0,
  "pair_seen_in_baseline": false,
  "src_baseline_max_ports_per_min": 1,
  "duration_minutes": 5.0
} 

▼ 규모 1위 사건 (case_id=69) — Claude가 받는 입력
{
  "active_minutes": 19,
  "n_flows": 230850,
  "peak_flows_per_min": 17752,
  "n_src_ips": 1,
  "n_dst_ips": 1,
  "n_dst_po

In [26]:
# 사건 단위 트리아지 (5건씩 동시에)

# v2-3. 사건 단위 트리아지 (5건씩 동시에)
from concurrent.futures import ThreadPoolExecutor
import threading
lock, done, failures_v2 = threading.Lock(), [0], {}
MAX_TOKENS = globals().get("MAX_TOKENS", 4000)

def parse_json(text):
    start = text.find("{")
    if start < 0:
        return None
    try:
        return json.JSONDecoder().raw_decode(text[start:])[0]
    except json.JSONDecodeError:
        return None

def triage_v2(row):
    key = f"v2|{ANTHROPIC_MODEL}|{row.case_id}|{row.start_ts}|{row.n_flows}"
    if key in cache:
        out = cache[key]
    else:
        out = {"verdict": "error"}
        payload = json.dumps(build_payload_v2(row), ensure_ascii=False)
        for attempt in range(3):
            resp = client.messages.create(model=ANTHROPIC_MODEL, max_tokens=MAX_TOKENS, system=SYSTEM_V2,
                                          messages=[{"role": "user", "content": payload}])
            text = "".join(b.text for b in resp.content if b.type == "text")
            parsed = parse_json(text)
            if isinstance(parsed, dict) and "verdict" in parsed:
                out = parsed
                with lock:
                    cache[key] = out
                    json.dump(cache, open(CACHE_PATH, "w"), ensure_ascii=False)
                break
            with lock:
                failures_v2[row.case_id] = dict(stop_reason=resp.stop_reason, tail=text[-300:])
    with lock:
        done[0] += 1
        print(f"\r{done[0]}/{len(cases)}", end="")
    return out

with ThreadPoolExecutor(max_workers=5) as ex:
    results_v2 = list(ex.map(triage_v2, [r for _, r in cases.iterrows()]))
print()

triaged_v2 = pd.concat([cases.reset_index(drop=True), pd.json_normalize(results_v2).add_prefix("llm_")], axis=1)
triaged_v2.to_csv(f"{OUT_DIR}/triaged_cases_v2.csv", index=False)
print(triaged_v2.llm_verdict.value_counts().to_string())
if failures_v2:
    print(f"\n실패 {len(failures_v2)}건:", next(iter(failures_v2.values())))

74/74
llm_verdict
malicious     59
suspicious    15


In [27]:
# 채점 + v1과 나란히 비교 (인시던트 105건 vs 사건 수십 건)

# v2-4. 채점 + v1과 비교
def score(df, rule_col):
    d = df.copy()
    d["is_attack_unit"] = d.attack_share >= 0.5
    d["truth"] = d.top_label.str.lower().map(TRUTH)
    d["id_valid"] = d.llm_technique_id.map(lambda x: isinstance(x, str) and x in ATTACK and not ATTACK[x]["retired"])
    d["llm_match"] = [parent(x) in s if isinstance(s, set) else np.nan for x, s in zip(d.llm_technique_id, d.truth)]
    d["rule_match"] = [RULE_BASELINE[r] in s if isinstance(s, set) else np.nan for r, s in zip(d[rule_col], d.truth)]
    d["conf"] = pd.to_numeric(d.llm_confidence, errors="coerce")
    att, fp = d[d.is_attack_unit], d[~d.is_attack_unit]
    named = d[d.llm_technique_id.map(lambda x: isinstance(x, str) and x != "")]
    missed = att[att.llm_verdict == "benign"]
    res = {
        "분석 단위 수": len(d),
        "기법 ID 유효율": named.id_valid.mean(),
        "공격 단위 수": len(att),
        "공격을 악성/의심으로 판정": (att.llm_verdict != "benign").mean(),
        "ATT&CK 매핑 정확도 (LLM)": att.llm_match.astype(float).mean(),
        "ATT&CK 매핑 정확도 (규칙명 고정)": att.rule_match.astype(float).mean(),
        "헛경보 단위 수": len(fp),
        "헛경보를 정상으로 걸러냄": (fp.llm_verdict == "benign").mean() if len(fp) else np.nan,
        "정상 판정으로 놓친 공격 플로우": int(missed.attack_flows.sum()) if "attack_flows" in d else np.nan,
    }
    for c in [0.7, 0.8]:
        closed = d[(d.llm_verdict == "benign") & (d.conf >= c)]
        res[f"자동종결 ≥{c}: 종결 건수"] = len(closed)
        res[f"자동종결 ≥{c}: 공격 잘못 종결"] = int(closed.is_attack_unit.sum())
    return d, res

# v1의 '놓친 공격 플로우'도 같은 기준으로 계산
v1 = triaged.copy()
v1["attack_flows"] = (v1.attack_share * v1.n_flows).round().astype(int)
s1, r1 = score(v1, "rule")
s2, r2 = score(triaged_v2, "primary_rule")

PCT = {"기법 ID 유효율", "공격을 악성/의심으로 판정", "ATT&CK 매핑 정확도 (LLM)",
       "ATT&CK 매핑 정확도 (규칙명 고정)", "헛경보를 정상으로 걸러냄"}
def fmt(k, v):
    if pd.isna(v): return "-"
    return f"{v:.0%}" if k in PCT else f"{int(v):,}"
comp = pd.DataFrame({"v1 (인시던트)": {k: fmt(k, v) for k, v in r1.items()},
                     "v2 (사건 + 맥락)": {k: fmt(k, v) for k, v in r2.items()}})
print(comp.to_string())

print("\n[v2] 공격 유형별 매핑 정확도")
print(s2[s2.is_attack_unit].groupby("top_label").agg(
    cases=("case_id", "count"), llm_acc=("llm_match", "mean"), rule_acc=("rule_match", "mean"),
    llm_ids=("llm_technique_id", lambda s: ", ".join(sorted(set(map(str, s)))))).to_string())

print("\n[v2] 공격인데 정상으로 판정")
print(s2[s2.is_attack_unit & (s2.llm_verdict == "benign")][
    ["case_id", "rules", "top_label", "n_flows", "conf", "llm_summary_ko"]].to_string())

print("\n[v2] 헛경보 사건 판정")
print(s2[~s2.is_attack_unit][["case_id", "rules", "n_flows", "top_dst_port", "llm_verdict",
                               "conf", "llm_summary_ko"]].to_string())

                       v1 (인시던트) v2 (사건 + 맥락)
분석 단위 수                      105           74
기법 ID 유효율                   100%         100%
공격 단위 수                       97           66
공격을 악성/의심으로 판정               90%         100%
ATT&CK 매핑 정확도 (LLM)          74%          97%
ATT&CK 매핑 정확도 (규칙명 고정)       68%          86%
헛경보 단위 수                       8            8
헛경보를 정상으로 걸러냄                75%           0%
정상 판정으로 놓친 공격 플로우        150,585            0
자동종결 ≥0.7: 종결 건수               8            0
자동종결 ≥0.7: 공격 잘못 종결            4            0
자동종결 ≥0.8: 종결 건수               3            0
자동종결 ≥0.8: 공격 잘못 종결            0            0

[v2] 공격 유형별 매핑 정확도
                               cases llm_acc rule_acc    llm_ids
top_label                                                       
DDoS                               1     1.0      1.0  T1499.002
DoS GoldenEye                      2     1.0      1.0  T1499.002
DoS Hulk                           1     1.0      1.0  T1499.002
DoS Slowhtt

In [ ]:
# 12단계: 확신도로 우선순위 큐 만들기

In [28]:
# 확신도 분포 + 우선순위 큐

# v2-5. 확신도로 공격과 헛경보가 갈리나? → 우선순위 큐로 활용
s2["group"] = np.where(s2.is_attack_unit, "공격", "헛경보")
print("판정 × 확신도 분포")
print(s2.groupby(["group", "llm_verdict"]).conf.describe()[["count", "min", "50%", "max"]].to_string())

def tier(r):
    if r.llm_verdict == "malicious" and r.conf >= 0.7: return "P1 즉시 대응"
    if r.llm_verdict == "benign" or (r.llm_verdict == "suspicious" and r.conf < 0.5): return "P3 후순위 검토"
    return "P2 일반 검토"
s2["tier"] = s2.apply(tier, axis=1)

print("\n우선순위 큐 (v2)")
print(s2.groupby("tier").agg(
    사건=("case_id", "count"),
    공격_사건=("is_attack_unit", "sum"),
    헛경보_사건=("is_attack_unit", lambda x: int((~x).sum())),
    공격_플로우=("attack_flows", "sum")).to_string())

판정 × 확신도 분포
                   count   min   50%   max
group llm_verdict                         
공격    malicious     57.0  0.60  0.90  0.93
      suspicious     9.0  0.40  0.50  0.60
헛경보   malicious      2.0  0.78  0.79  0.80
      suspicious     6.0  0.30  0.35  0.50

우선순위 큐 (v2)
           사건  공격_사건  헛경보_사건    공격_플로우
tier                                  
P1 즉시 대응   53     51       2  227626.0
P2 일반 검토   13     12       1  380435.0
P3 후순위 검토   8      3       5    5729.0


In [29]:
# P3에 섞인 공격 3건 확인 (마지막 확인 셀)
print(s2[(s2.tier == "P3 후순위 검토") & s2.is_attack_unit][
    ["case_id", "rules", "top_label", "n_flows", "llm_verdict", "conf", "llm_summary_ko"]].to_string())

    case_id                         rules                 top_label  n_flows llm_verdict  conf                                                                                                                                                                                                                                  llm_summary_ko
64       65  R3_VOLUME_FLOOD, R4_SLOW_DOS          DoS Slowhttptest      343  suspicious   0.4                 방화벽 내부 인터페이스(int-01, NAT)에서 웹서버(int-02) 80번 포트로 약 2분간 343개의 짧은 데이터·긴 지속시간(중앙값 약 21초) 연결이 발생했으며 기준선에는 없던 트래픽입니다. 서비스 고갈형 DoS(느린 HTTP 등) 가능성이 있으나 NAT 때문에 실제 출발지를 알 수 없고 정상 트래픽 급증일 수도 있어 의심 수준으로 판단하며, NAT 로그와 웹서버 로그 확인이 필요합니다.
70       71  R3_VOLUME_FLOOD, R4_SLOW_DOS             DoS GoldenEye     4123  suspicious   0.4     방화벽 내부 인터페이스(int-01, NAT 주소)에서 웹 서버(int-02)의 80 포트로 2~3분간 4123개의 플로우(분당 최대 2017건)가 발생했으며, 기준 하루 동안에는 없던 트래픽 패턴입니다. 다만 대부분의 연결이 정상적으로 응답을 받고 완료되어 HTTP 다운로드 형태를 보이므로 DoS 공격과 크롤러·부하 테스트·정상적인 접속 급증을 구분하기 위해 실제 외부 IP와 웹 로그에 대한 추가 확인이 필요